#### Model Training pipeline

In [1]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor

import sys
sys.executable


os.chdir("/Users/ARahim/Documents/GitHub/eliza-deploy")

def remove_outliers(df, columns, tolerance=1.5):
    """
    Remove outliers from the DataFrame based on the specified columns using IQR.
    Args:
    df (DataFrame): Input DataFrame.
    columns (list): List of column names to check for outliers.
    tolerance (float): Tolerance factor for outlier removal using IQR. Default is 1.5.
    Returns:
    DataFrame: DataFrame with outliers removed.
    """
    for column in columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - tolerance * iqr
        upper_bound = q3 + tolerance * iqr
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df

# Load the CSV file into a DataFrame
file_path = "data/properties.csv"
properties_df = pd.read_csv(file_path)

# Split the DataFrame based on the 'property_type' column
df_house = properties_df[properties_df['property_type'] == 'HOUSE']

# Split df_house into training and testing sets
train_house, test_house = train_test_split(df_house, test_size=0.2, random_state=42)

# Define columns for outlier removal
columns_for_outlier_removal = ["price", "total_area_sqm", "surface_land_sqm", "nbr_frontages", 
                               "nbr_bedrooms", "terrace_sqm", "garden_sqm", 
                               "primary_energy_consumption_sqm"]

# Remove outliers from training set
train_house_no_outliers = remove_outliers(train_house, columns_for_outlier_removal)

# Remove outliers from test set (based on training set's statistics)
test_house_no_outliers = remove_outliers(test_house, columns_for_outlier_removal)

# Define numerical and categorical features
num_features = [ "total_area_sqm", "surface_land_sqm", "nbr_frontages", 
                "nbr_bedrooms", "terrace_sqm", "garden_sqm", 
                 "primary_energy_consumption_sqm",
                "zip_code", "latitude", "longitude"]

fl_features = ["fl_furnished", "fl_open_fire", "fl_terrace", "fl_garden", "fl_swimming_pool", "fl_floodzone",
               "fl_double_glazing"]
cat_features = ["subproperty_type", "region", "province", "locality", 
                "equipped_kitchen", "state_building", "epc", "heating_type"]

# Define preprocessor with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(categories='auto', handle_unknown='ignore'))  # Adjusted OneHotEncoder
        ]), cat_features)
    ])

# Define model
model = GradientBoostingRegressor()

# Define the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Fit the pipeline on training data
pipeline.fit(train_house_no_outliers.drop(columns=['price']), train_house_no_outliers['price'])

# Evaluate the model
train_score = pipeline.score(train_house_no_outliers.drop(columns=['price']), train_house_no_outliers['price'])
test_score = pipeline.score(test_house_no_outliers.drop(columns=['price']), test_house_no_outliers['price'])

print("Training score:", train_score)
print("Test score:", test_score)

# Perform cross-validation
cv_scores = cross_val_score(pipeline, train_house_no_outliers.drop(columns=['price']), train_house_no_outliers['price'], cv=5)
print("Cross-validation scores:", cv_scores)

# Define the path to save the model
model_folder = "api"
model_path = os.path.join(model_folder, "model.pkl")

# Save the model to a file using pickle
with open(model_path, 'wb') as f:
    pickle.dump(pipeline, f)

Training score: 0.7927104319453137
Test score: 0.7512161606243928
Cross-validation scores: [0.73808608 0.747828   0.75996022 0.74689865 0.75775133]
